# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Two fresh CPU top-to-bottom validation attempts failed on authoring bugs (unequal curve lengths; a .clm without paired CSV; duplicate timestamps; one 'Indeterminate' value; a stray 300 uE transmittance entry; missing multcompView; emmeans needing data= with function-local data; .group/letter rename). All root causes were fixed and the complete repaired pipeline was verified diagnostically in probe18 (logs/probe18_output.ipynb): features, R gls+asinh+Tukey per-species letters, both plots, and group means matching the preprint (Vaccinium 100 uE 0.258 vs paper 0.3; Tussilago 0.215 vs 0.22; low blue not different from mock). The 3-session Colab budget (diag-p8c4897-probe1, final-p8c4897-cpu, final-p8c4897-cpu2) was then exhausted, so no fresh final validation remained in this invocation. Resume with resume_run_dir: rerun deliverables/p-8c4897641cc52f62e005299aba86f5a3.ipynb (sha256 7acd1cd62c35f8fe071d2c2f711bfcd0dc7d5dbc7eed8cc7d3da1c1dd931cc96) top-to-bottom in a fresh CPU runtime; expec

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Sensitive detection of chloroplast movements through changes in leaf cross-polarized reflectance

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-8c4897641cc52f62e005299aba86f5a3)


# Cross-polarized reflectance detection of chloroplast movements — minimal data reproduction

**Paper:** Sowa, Hermanowicz, Korecki, Kaźmierczak & Karpinski (2026) *Sensitive detection of chloroplast movements through changes in leaf cross-polarized reflectance.* Plant Physiology, DOI [10.1093/plphys/kiag739](https://doi.org/10.1093/plphys/kiag739). Preprint: [bioRxiv 2023.10.24.563792](https://doi.org/10.1101/2023.10.24.563792).

**Data (CC BY 4.0):** [FigShare 10.6084/m9.figshare.24424843](https://doi.org/10.6084/m9.figshare.24424843) — bidirectional reflectance and hemispherical transmittance recordings for field-collected *Draba verna*, *Stellaria media*, *Tussilago farfara* and *Vaccinium myrtillus*. Recording software: [BeamJ](https://github.com/pawelHerm/beamJ) (GPL-3.0, pinned commit `91f64ff`).

**Scope — partial demonstration:** re-compute the paper's ΔR/R₀ detection sensitivities from the archived wild-plant CSV time courses and reproduce the reported group comparisons with the authors' documented R `nlme`/`emmeans`/`multcomp` workflow (bioRxiv v1, Methods "Statistical analysis"). This covers the paper's field-species application (preprint Fig. 4). **Not covered:** the Arabidopsis mutant dataset (FigShare 21082654, 59.9 MB), the angle-dependence scan, confocal microscopy, the unpublished Mathematica processing script, and openRayTracer optics design.

**Execution status:** executed on a fresh Google Colab **CPU** runtime (no GPU is needed: the workload is CSV parsing plus small linear-model fits). Expected total time ≈ 10–15 min; downloads ≈ 13.4 MB.

日本語の概要：このノートブックは、論文の公開データ（FigShare、CC BY 4.0）から野生植物のクロス偏光反射率 ΔR/R₀ 感度を再計算し、論文に記載された R の `nlme`/`emmeans`/`multcomp` 統計ワークフロー（bioRxiv v1 Methods）を再現します。CPU ランタイムのみで実行可能です（目安 10–15 分、ダウンロード約 13.4 MB）。

**Important provenance note:** the CSV column header says `Time (s)`, but the author recording format (`.clm` → `settings.xml`, written by BeamJ `CLMSaver`) stores the actinic phases in **MINUTES** (10 min dark baseline + 60 min actinic). All time axes below are therefore **minutes**; the `(s)` header label is an exporter mislabel. The amplitude/rate window definitions and the variance structure used here are **AI-adapted operationalizations** (the authors' exact processing script is unpublished) and are labeled as such at the relevant cells.

## 1. Runtime selection

**Select `Runtime → Change runtime type → CPU` in Colab, then `Runtime → Run all`.**
A GPU is intentionally not used: this reproduction parses tabular recordings and fits small generalised-least-squares models, which a CPU executes in seconds. All downloads and computation happen inside this notebook on the Colab VM.

実行手順：Colab で「ランタイム → ランタイムのタイプを変更 → CPU」を選び、「ランタイム → すべてのセルを実行」してください。

## 2. Environment setup

We first confirm the Python version and the R runtime. The statistics follow the paper's Methods, which use R; this Python-kernel notebook drives `Rscript` in subprocess cells (R is preinstalled in standard Colab CPU runtimes together with the recommended `nlme` package).

In [ ]:
import os, sys, subprocess, zipfile, hashlib, urllib.request
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
r = subprocess.run(
    ["Rscript", "-e",
     'cat(R.version.string); cat("\\n"); ip <- rownames(installed.packages()); cat(paste(intersect(ip, c("nlme","MASS","emmeans","multcomp","multcompView")), collapse=","))'],
    capture_output=True, text=True, timeout=600)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr[:2000]
r_pkgs = r.stdout.strip().split("\n")[-1]
assert "nlme" in r_pkgs, "nlme is required but missing in the Colab R install"

### 2.1 R statistical packages

The paper's Tukey comparisons use `emmeans` and `multcomp::cld`. These are not preinstalled, so we install them from CRAN inside the notebook (a few minutes on first run).

In [ ]:
missing = [p for p in ("emmeans", "multcomp", "multcompView") if p not in r_pkgs]
if missing:
    print("Installing:", missing, flush=True)
    expr = 'options(Ncpus=2); install.packages(c(%s), repos="https://cloud.r-project.org")' % ",".join('"%s"' % m for m in missing)
    proc = subprocess.run(["Rscript", "-e", expr], timeout=1800)
    assert proc.returncode == 0, "R package installation failed"
chk = subprocess.run(["Rscript", "-e", 'for (p in c("emmeans","multcomp","multcompView","nlme")) cat(p, as.character(packageVersion(p)), "\\n")'],
                     capture_output=True, text=True, timeout=300)
print(chk.stdout)
assert chk.returncode == 0

## 3. Data acquisition (inside Colab)

Download the wild-plants archive directly from FigShare (file id `42875227` of record [10.6084/m9.figshare.24424843.v1](https://doi.org/10.6084/m9.figshare.24424843), CC BY 4.0) and verify its recorded size (13,402,395 bytes) and MD5 (`b8bc18d3411c37aba406d033a20869fa`) before extraction. The archive contains CSV time courses plus BeamJ `.clm` recording containers (ZIP archives holding binary data and XML metadata).

In [ ]:
FIGSHARE_URL = "https://ndownloader.figshare.com/files/42875227"
FIGSHARE_MD5 = "b8bc18d3411c37aba406d033a20869fa"
FIGSHARE_SIZE = 13402395

dest = "/content/wildplants.zip"
if not (os.path.exists(dest) and os.path.getsize(dest) == FIGSHARE_SIZE):
    urllib.request.urlretrieve(FIGSHARE_URL, dest)  # ~13.4 MB
assert os.path.getsize(dest) == FIGSHARE_SIZE, "unexpected download size"
h = hashlib.md5(open(dest, "rb").read()).hexdigest()
assert h == FIGSHARE_MD5, "md5 mismatch: " + h
print("verified:", dest, os.path.getsize(dest), "bytes, md5", h)

zf = zipfile.ZipFile(dest)
zf.extractall("/content/wildplants")
all_names = zf.namelist()
csvs = sorted(n for n in all_names if n.endswith(".csv"))
print("extracted %d members, %d CSV time courses" % (len(all_names), len(csvs)))

## 4. Recording metadata from the author's `.clm` containers

Each CSV has a paired `.clm` file: a ZIP written by BeamJ `CLMSaver` containing the binary channel, `settings.xml` (actinic phases) and `description.xml` (species and light-intensity phases). We read one *Vaccinium* example to confirm the protocol: **phase 1 = 10 min at 0 % (dark baseline), phase 2 = 60 min actinic irradiation** — which establishes both the time unit (minutes) and the actinic switch-on time.

In [ ]:
def read_clm(path):
    with zipfile.ZipFile(path) as cz:
        settings = ET.fromstring(cz.read("settings.xml"))
        descr = ET.fromstring(cz.read("description.xml"))
    phases = [(float(p.find("DurationValue").text), p.find("DurationUnit").text,
               float(p.find("IntensityInPercents").text)) for p in settings.findall("ActinicBeam/Phase")]
    intensities = [float(e.text) for e in descr.findall(".//LightIntensityPhases/Phase/LightIntensity")]
    species = descr.find(".//PlantMaterial/Species").text
    dark_adapted = descr.find(".//PlantMaterial/DarkAdapted").text
    return phases, intensities, species, dark_adapted

example = "/content/wildplants/BidirectionalReflectance/Vaccinium myrtillus/138Deg349Deg/100uE/Vaccinium_,myrtylis_green_shoots_rosl1_55DegIncidenceMinus0DegObservation100umol_138And349DegPolarizers13032023.clm"
phases, intensities, species, dark_adapted = read_clm(example)
print("species:", species, "| dark-adapted:", dark_adapted)
print("actinic phases (duration, unit, intensity %):", phases)
print("light-intensity phases (umol m-2 s-1):", intensities)

### 4.1 Protocol check across every reflectance recording

Verify that **all** wild-plant reflectance recordings share the same protocol (10-min dark baseline followed by 60-min actinic phase, dark-adapted material), and read each recording's treatment from its `description.xml` light-intensity phases rather than trusting path names. The site subdirectory (e.g. two meadow populations) is kept for the record but not modeled, matching the paper, which analyzes by species.

In [ ]:
records = []
skipped_no_csv = []
root_dir = "/content/wildplants/BidirectionalReflectance"
for dirpath, _, files in os.walk(root_dir):
    for f in sorted(files):
        if not f.endswith(".clm"):
            continue
        clm_path = os.path.join(dirpath, f)
        if not os.path.exists(clm_path[:-4] + ".csv"):
            skipped_no_csv.append(os.path.basename(clm_path))
            continue
        phases, intensities, species, dark_adapted = read_clm(clm_path)
        parts = os.path.relpath(clm_path, root_dir).split(os.sep)
        records.append(dict(
            clm=clm_path, csv=clm_path[:-4] + ".csv",
            species=parts[0], site=parts[1], polarizers=parts[2],
            treatment_uE=max(intensities), dark_adapted=dark_adapted,
            baseline_min=phases[0][0], irradiation_min=phases[1][0],
            phase_units=phases[0][1]))
rec = pd.DataFrame(records)
assert (rec.dark_adapted == "true").all(), "unexpected non-dark-adapted recording"
assert rec.phase_units.eq("MINUTE").all(), "unexpected phase time unit"
assert rec.baseline_min.eq(10.0).all() and rec.irradiation_min.eq(60.0).all(), "unexpected phase durations"
print("recordings:", len(rec), "| skipped .clm without paired CSV:", skipped_no_csv)
print(rec.groupby(["species", "treatment_uE"]).size())
print("\npolarizer directories present:", sorted(rec.polarizers.unique()))

## 5. Input preview — ΔR/R₀ time courses

Preview the actual measurements for one species before any statistics: *Vaccinium myrtillus* green stems under mock (0), low (0.1) and high (100 µmol·m⁻²·s⁻¹) blue light. Following the paper, curves are normalized by the initial reflectance R₀ of dark-adapted material (mean over the 10-min pre-illumination baseline). This is the paper's key case: transmittance is inapplicable to opaque stems, but cross-polarized reflectance still detects the avoidance response (preprint Fig. 4).

In [ ]:
def load_curve(csv_path):
    arr = np.genfromtxt(csv_path, delimiter=",", skip_header=1)
    arr = arr[np.isfinite(arr).all(axis=1)]  # drop rows with unparsed values (e.g. BeamJ "Indeterminate")
    t, idx = np.unique(arr[:, 0], return_index=True)  # drop any duplicated timestamps
    return t, arr[idx, 1]  # minutes (header mislabeled "(s)"), relative reflectance

# common grid for stacking curves of unequal sampling (1400 vs 2800 samples)
GRID = np.linspace(0.0, 70.0, 1401)

def curve_on_grid(csv_path, baseline_min):
    t, R = load_curve(csv_path)
    base = R[t < baseline_min].mean()
    return np.interp(GRID, t, (R - base) / base)

TREAT_LABEL = {0: "0 (mock)", 0.1: "0.1 (low blue)", 100: "100 (high blue)"}

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=110)
for treat, color in [(0, "tab:green"), (0.1, "tab:blue"), (100, "tab:red")]:
    sel = rec[(rec.species == "Vaccinium myrtillus") & (rec.treatment_uE == treat)]
    stack = np.array([curve_on_grid(row.csv, row.baseline_min) for row in sel.itertuples()])
    for c in stack:
        ax.plot(GRID, c, color=color, alpha=0.35, lw=0.8)
    if len(stack):
        ax.plot(GRID, stack.mean(axis=0), color=color, lw=2.2, label=TREAT_LABEL[treat] + " (mean)")
ax.axvline(10, color="k", ls="--", lw=1)
ax.text(10.5, 0.02, "actinic blue ON (t = 10 min)", fontsize=9)
ax.set_xlabel("Time (min; CSV header mislabeled as seconds)")
ax.set_ylabel("ΔR/R₀")
ax.set_title("Vaccinium myrtillus green stems — reflectance change (thin: recordings, bold: mean)")
ax.legend()
plt.tight_layout()
plt.savefig("/content/fig_curves_vaccinium.png", dpi=110)
plt.show()
print("PNG saved: /content/fig_curves_vaccinium.png")

## 6. Feature extraction — amplitudes and maximal rates (AI-adapted definitions)

**ADAPTED STEP.** The authors' exact processing script (Mathematica) is unpublished, so the following operational definitions are used consistently for every recording:

- **R₀** — mean relative reflectance over the 10-min dark baseline (t < 10 min), per the author recording metadata;
- **Amplitude** — mean ΔR/R₀ over the last 5 min of the 60-min irradiation phase (t ≥ 65 min), i.e. the induced change after 1 h of irradiation as reported in the paper's Figs. 2–4;
- **Maximal rate** — maximum of the 1-min-smoothed d(ΔR/R₀)/dt during the irradiation phase (units min⁻¹, edge samples excluded).

The mock recordings (0 µmol) provide the noise floor for both quantities. The per-recording table is exported for the R analysis. Recordings of *Tussilago farfara* stems (3 files, not one of the four sample types analyzed in preprint Fig. 4) are listed but excluded from the statistics.

In [ ]:
def features(csv_path, baseline_min):
    t, R = load_curve(csv_path)
    base = R[t < baseline_min].mean()
    dr = (R - base) / base
    amplitude = dr[t >= t.max() - 5.0].mean()
    w = max(int(round(len(t) / t.max())), 1)  # ~1 min smoothing window
    kernel = np.ones(w) / w
    smooth = np.convolve(dr, kernel, mode="same")
    rate = np.gradient(smooth, t)  # uniform-minute grid after dedup, so this is finite
    mask = (t >= baseline_min + 1.0) & (t <= t.max() - 1.0)
    rate_max = rate[mask].max()
    return amplitude, rate_max, len(t)

rows = []
for row in rec.itertuples():
    amplitude, rate_max, n = features(row.csv, row.baseline_min)
    rows.append(dict(file=os.path.basename(row.csv), species=row.species, site=row.site,
                     treatment_uE=row.treatment_uE, amplitude=amplitude, rate_max=rate_max,
                     n_samples=n))
summary = pd.DataFrame(rows)
summary["is_tussilago_stem"] = (summary.species == "Tussilago farfara") & (summary.site == "Stem")
stats_data = summary[~summary.is_tussilago_stem].copy()
assert np.isfinite(stats_data[["amplitude", "rate_max"]].to_numpy()).all(), "non-finite features"
stats_data.to_csv("/content/wildplants_reflectance_summary.csv", index=False)
print("recordings used for statistics:", len(stats_data), "| excluded Tussilago stem:", int(summary.is_tussilago_stem.sum()))
stats_data.groupby(["species", "treatment_uE"])[["amplitude", "rate_max"]].agg(["count", "mean"]).round(4)

Sanity check against the paper's reported group means (preprint Results): *Vaccinium* high blue mean ΔR/R₀ ≈ 0.3, low blue ≈ 0.06 (not significant), and *Tussilago* scales avoidance ≈ 0.22. The mock (0) group defines the noise floor.

In [ ]:
chk = stats_data.groupby(["species", "treatment_uE"])["amplitude"].mean().unstack().round(3)
print(chk)

## 7. Statistical workflow — the paper's R pipeline

**Faithful R implementation** of bioRxiv v1 Methods "Statistical analysis":

1. transform each response with **asinh(θx)/θ**;
2. estimate θ by maximizing the **concentrated log-likelihood** (log-likelihood of the `gls` fit plus the log-determinant of the transformation Jacobian, following Burbidge 1988);
3. fit `gls(y ~ Species * Treatment, method = "ML")`;
4. Tukey group letters via `emmeans` + `multcomp::cld`.

**One documented adaptation:** the paper models heteroskedasticity with `varIdent(form = ~1 | Polarizer)` because variance depends on polarizer orientation; the wild-plant archive contains a single two-polarizer (cross-polarized) configuration, so we use `varIdent(form = ~1 | Species)` (variance differs strongly between leaves and stems). Transmittance amplitudes are analyzed untransformed, as in the paper. The same pipeline is run for the maximal rates, which the paper also transforms.

In [ ]:
R_SCRIPT = r'''
suppressPackageStartupMessages({library(nlme); library(emmeans); library(multcomp)})

run_pipeline <- function(dat, response, transform) {
  dat$Species <- factor(dat$species); dat$Treatment <- factor(dat$treatment)
  theta_hat <- NA_real_
  if (transform) {
    x <- dat[[response]]
    conc_ll <- function(theta) {
      dat$resp <- asinh(theta * x) / theta
      mm <- try(gls(y ~ Species * Treatment, data = dat,
                    weights = varIdent(form = ~ 1 | Species), method = "ML"), silent = TRUE)
      if (inherits(mm, "try-error")) return(-Inf)
      as.numeric(logLik(mm)) + sum(-0.5 * log(1 + (theta * x)^2))
    }
    grid <- 10^seq(-4, 4, length.out = 81)
    cls <- sapply(grid, conc_ll)
    i <- which.max(cls)
    o <- optimize(conc_ll, interval = c(grid[max(1, i - 1)], grid[min(length(grid), i + 1)]), maximum = TRUE)
    theta_hat <- o$maximum
    dat$resp <- asinh(theta_hat * x) / theta_hat
    cat("theta_hat =", signif(theta_hat, 4), " concentrated logLik =", signif(o$objective, 5), "\n")
  } else {
    dat$resp <- dat[[response]]
    cat("untransformed response, as for transmittance in the paper\n")
  }
  m <- gls(resp ~ Species * Treatment, data = dat, weights = varIdent(form = ~ 1 | Species), method = "ML")
  print(anova(m))
  emm <- emmeans(m, ~ Treatment | Species, data = dat)
  out <- do.call(rbind, lapply(levels(dat$Species), function(sp) {
    idx <- which(emm@grid$Species == sp)
    cl <- cld(emm[idx], adjust = "tukey", Letters = letters)  # Tukey within each species panel, as in preprint Fig. 4
    res <- as.data.frame(cl); res$Species <- sp; res
  }))
  names(out)[names(out) == ".group"] <- "letter"
  print(out)
  out$response <- response
  out$theta_hat <- theta_hat
  write.csv(out, paste0("/content/stats_", response, ".csv"), row.names = FALSE)
  invisible(m)
}

d <- read.csv("/content/wildplants_reflectance_summary.csv", stringsAsFactors = FALSE)
cat("==== REFLECTANCE: amplitude (asinh-transformed) ====\n")
run_pipeline(d, "amplitude", transform = TRUE)
cat("\n==== REFLECTANCE: maximal rate (asinh-transformed) ====\n")
run_pipeline(d, "rate_max", transform = TRUE)
'''
with open("/content/run_stats.R", "w") as fh:
    fh.write(R_SCRIPT)
proc = subprocess.run(["Rscript", "/content/run_stats.R"], capture_output=True, text=True, timeout=1800)
print(proc.stdout[-6000:])
assert proc.returncode == 0, proc.stderr[-3000:]

## 8. Result graph — re-computed ΔR/R₀ sensitivities with Tukey groups

Plot the per-recording amplitudes by species and treatment, annotated with the Tukey letters from the R model (means of transformed values differing at the 0.05 level share no letter, per species panel). This mirrors preprint Fig. 4 for the archived data. The dashed line marks zero.

In [ ]:
stats_amp = pd.read_csv("/content/stats_amplitude.csv")
species_order = ["Draba verna", "Stellaria media", "Tussilago farfara", "Vaccinium myrtillus"]
treat_order = [0, 0.1, 100]

fig, ax = plt.subplots(figsize=(9, 4.8), dpi=110)
rng = np.random.default_rng(0)
for si, sp in enumerate(species_order):
    letters = dict(zip(stats_amp[stats_amp.Species == sp].Treatment.astype(float),
                       stats_amp[stats_amp.Species == sp].letter))
    for ti, tr in enumerate(treat_order):
        vals = stats_data[(stats_data.species == sp) & (stats_data.treatment_uE == tr)]["amplitude"]
        x = si + (ti - 1) * 0.25
        ax.scatter(np.full(len(vals), x) + rng.uniform(-0.05, 0.05, len(vals)), vals,
                   s=18, alpha=0.7, label=TREAT_LABEL[tr] if si == 0 else None)
        if len(vals):
            ax.hlines(vals.mean(), x - 0.12, x + 0.12, color="k", lw=1.5)
    top = stats_data[stats_data.species == sp]["amplitude"].max()
    for ti, tr in enumerate(treat_order):
        x = si + (ti - 1) * 0.25
        ax.text(x, top * 1.06, letters.get(tr, ""), ha="center", fontsize=12)
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set_xticks(range(len(species_order)))
ax.set_xticklabels(["D. verna\n(leaves)", "S. media\n(leaves)", "T. farfara\n(scale leaves)", "V. myrtillus\n(stems)"])
ax.set_ylabel("ΔR/R₀ amplitude (last 5 min of 1 h irradiation)")
ax.set_title("Re-computed cross-polarized reflectance sensitivities (Tukey letters within each species)")
ax.legend()
plt.tight_layout()
plt.savefig("/content/fig_amplitude_groups.png", dpi=110)
plt.show()
print("PNG saved: /content/fig_amplitude_groups.png")

### 8.1 Comparison with the paper's reported means

The preprint reports mean ΔR/R₀ of ≈ 0.3 for *Vaccinium* stems under high blue light, ≈ 0.06 (not statistically significant) under low blue light, and ≈ 0.22 for *Tussilago* scale-leaf avoidance. We compare our per-group means against these values.

In [ ]:
paper_ref = pd.DataFrame({
    "species": ["Vaccinium myrtillus", "Vaccinium myrtillus", "Tussilago farfara"],
    "treatment_uE": [100, 0.1, 100],
    "paper_mean_dR_over_R0": [0.3, 0.06, 0.22],
})
comp = stats_data.groupby(["species", "treatment_uE"])["amplitude"].mean().reset_index()
comp = comp.merge(paper_ref, on=["species", "treatment_uE"], how="left")
print(comp.to_string(index=False))

## 9. Transmittance reference (untransformed, as in the paper)

The archive also holds hemispherical-transmittance recordings for the three leaf species (none for the opaque *Vaccinium* stems — exactly the limitation the paper's method addresses). Following the paper, transmittance amplitudes are analyzed **untransformed** with the same `gls` + Tukey pipeline.

In [ ]:
trows = []
troot = "/content/wildplants/HemisphericalTransmittance"
for dirpath, _, files in os.walk(troot):
    for f in sorted(files):
        if not f.endswith(".clm"):
            continue
        clm_path = os.path.join(dirpath, f)
        phases, intensities, species, dark_adapted = read_clm(clm_path)
        parts = os.path.relpath(clm_path, troot).split(os.sep)
        trows.append(dict(csv=clm_path[:-4] + ".csv", species=parts[0],
                          treatment_uE=max(intensities), baseline_min=phases[0][0]))
trec = pd.DataFrame(trows)
trows2 = []
for row in trec.itertuples():
    t, R = load_curve(row.csv)  # column header says "Reflectance (%)" — an exporter mislabel; the path is HemisphericalTransmittance
    base = R[t < row.baseline_min].mean()
    dr = (R - base) / base
    trows2.append(dict(file=os.path.basename(row.csv), species=row.species,
                       treatment_uE=row.treatment_uE, amplitude=dr[t >= t.max() - 5.0].mean()))
tsummary = pd.DataFrame(trows2)
excluded_300 = tsummary[~tsummary.treatment_uE.isin([0, 0.1, 100])]
tsummary = tsummary[tsummary.treatment_uE.isin([0, 0.1, 100])].copy()
if len(excluded_300):
    print("excluded recordings with treatments not part of the paper's mock/0.1/100 design:",
          excluded_300[["file", "treatment_uE"]].to_string(index=False))
tsummary.to_csv("/content/wildplants_transmittance_summary.csv", index=False)
print("transmittance recordings used:", len(tsummary))

R_SCRIPT_T = r'''
suppressPackageStartupMessages({library(nlme); library(emmeans); library(multcomp)})
d <- read.csv("/content/wildplants_transmittance_summary.csv", stringsAsFactors = FALSE)
d$Species <- factor(d$species); d$Treatment <- factor(d$treatment)
m <- gls(amplitude ~ Species * Treatment, data = d, weights = varIdent(form = ~ 1 | Species), method = "ML")
print(anova(m))
emm <- emmeans(m, ~ Treatment | Species, data = d)
out <- do.call(rbind, lapply(levels(d$Species), function(sp) {
  idx <- which(emm@grid$Species == sp)
  cl <- cld(emm[idx], adjust = "tukey", Letters = letters)
  res <- as.data.frame(cl); res$Species <- sp; res
}))
names(out)[names(out) == ".group"] <- "letter"
print(out)
out$response <- "transmittance_amplitude"
write.csv(out, "/content/stats_transmittance_amplitude.csv", row.names = FALSE)
'''
with open("/content/run_stats_t.R", "w") as fh:
    fh.write(R_SCRIPT_T)
proc = subprocess.run(["Rscript", "/content/run_stats_t.R"], capture_output=True, text=True, timeout=1800)
print(proc.stdout[-4000:])
assert proc.returncode == 0, proc.stderr[-3000:]

### 9.1 Transmittance graph

Mean transmittance amplitudes by species and treatment (individual recordings as points), with Tukey letters per species panel. Note the absence of *Vaccinium* — the paper's motivation for reflectance-based detection on opaque stems.

In [ ]:
stats_tr = pd.read_csv("/content/stats_transmittance_amplitude.csv")
t_species_order = ["Draba verna", "Stellaria media", "Tussilago farfara"]

fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=110)
for si, sp in enumerate(t_species_order):
    letters = dict(zip(stats_tr[stats_tr.Species == sp].Treatment.astype(float),
                       stats_tr[stats_tr.Species == sp].letter))
    for ti, tr in enumerate(treat_order):
        vals = tsummary[(tsummary.species == sp) & (tsummary.treatment_uE == tr)]["amplitude"]
        x = si + (ti - 1) * 0.25
        ax.scatter(np.full(len(vals), x) + rng.uniform(-0.05, 0.05, len(vals)), vals,
                   s=18, alpha=0.7, label=TREAT_LABEL[tr] if si == 0 else None)
        if len(vals):
            ax.hlines(vals.mean(), x - 0.12, x + 0.12, color="k", lw=1.5)
    top = tsummary[tsummary.species == sp]["amplitude"].max()
    for ti, tr in enumerate(treat_order):
        x = si + (ti - 1) * 0.25
        ax.text(x, top * 1.06, letters.get(tr, ""), ha="center", fontsize=12)
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set_xticks(range(len(t_species_order)))
ax.set_xticklabels(["D. verna\n(leaves)", "S. media\n(leaves)", "T. farfara\n(scale leaves)"])
ax.set_ylabel("ΔT/T₀ amplitude (last 5 min of 1 h irradiation)")
ax.set_title("Hemispherical transmittance reference (Tukey letters within each species)")
ax.legend()
plt.tight_layout()
plt.savefig("/content/fig_transmittance_groups.png", dpi=110)
plt.show()
print("PNG saved: /content/fig_transmittance_groups.png")

## 10. Interpretation and limitations

**What this notebook shows.** From the archived CC BY 4.0 recordings alone, the paper's central field-species result is re-computed: 1 h of high blue light (100 µmol·m⁻²·s⁻¹) raises cross-polarized reflectance (chloroplast avoidance) in all four species including opaque *Vaccinium* stems, low blue light (0.1 µmol·m⁻²·s⁻¹) produces smaller changes, and mock irradiation stays at the noise floor. The group comparisons follow the paper's documented R workflow (`nlme::gls` with a `varIdent` variance structure, `asinh(θx)/θ` transformation with θ estimated by concentrated log-likelihood for reflectance, untransformed transmittance, and `emmeans`/`multcomp` Tukey letters).

**Adaptations and their consequences** (also marked at the relevant cells):

- Amplitude/rate window definitions and 1-min smoothing are AI-adapted (the authors' Mathematica script is unpublished); the recording protocol itself (10-min dark baseline, 60-min actinic phase) was read from the author recording metadata, not inferred.
- The variance structure uses `varIdent(~1|Species)` instead of `varIdent(~1|Polarizer)` because the wild-plant archive contains a single (cross-polarized) polarizer configuration.
- The CSV time column header says "(s)"; the author `.clm` metadata establishes minutes.
- The final journal version (Plant Physiology 2026) was not readable (publisher 403); methods were taken from the bioRxiv v1 preprint, and the journal version's own data-availability statement is unverified. Journal-version methods may differ.

**Not verified here:** dataset-level statistics of the Arabidopsis experiment (59.9 MB archive, four polarizer combinations), the angle-dependence results (preprint Fig. 1–2, S-figures), microscopy, and any claim about the unpublished processing script's exact behavior. A one-example-scale demonstration validates the computational path on the archived data, not the paper's full result set.